## Activity 1: Data Acquisition and Computational Analysis

## Task 1 — Define the Question and Unit of Analysis

##### 1.1 Research Question

How does building type relate to Site Energy Use Intensity
(Site EUI) among benchmarked buildings in Seattle?

##### 1.2 Unit of Analysis

One row represents one building-year benchmarking record.

A building may appear in multiple years because the dataset
contains repeated annual observations.

##### 1.3 Key Variables

- OSEBuildingID — building identifier
- DataYear — reporting year
- BuildingType — building classification
- SiteEUI(kBtu/sf) — site energy use intensity
- PropertyGFATotal — total property floor area
- ENERGYSTARScore — energy performance score, when available

##### 1.4 Population, Parameter, Sample, and Statistic

- Population: Building-year records represented by the Seattle
  Building Energy Benchmarking program for the years covered.
- Parameter: The true population-level relationship between
  building type and Site EUI.
- Sample: The 38,309 building-year records in the acquired dataset.
- Statistic: Sample summaries such as mean, median, and
  distribution of Site EUI by building type.


## Task 2 — Document the Source

In [28]:
source_information = pd.DataFrame({
    "Item": [
        "Dataset Name",
        "Data Provider",
        "Dataset ID",
        "Source Website",
        "API Endpoint",
        "Data Format",
        "Access Method",
        "Data Coverage",
        "Unit of Observation"
    ],
    "Description": [
        "Building Energy Benchmarking Data 2015–Present",
        "City of Seattle",
        "teqw-tu6e",
        "https://data.seattle.gov/Built-Environment/Building-Energy-Benchmarking-Data-2015-Present/teqw-tu6e",
        "https://data.seattle.gov/api/v3/views/teqw-tu6e/query.json",
        "CSV and JSON",
        "Public dataset accessed through API",
        "2015–2025 in the acquired dataset",
        "Building-year benchmarking record"
    ]
})

display(source_information)


,Item,Description
0,Dataset Name,Building Energy Benchmarking Data 2015–Present
1,Data Provider,City of Seattle
2,Dataset ID,teqw-tu6e
3,Source Website,https://data.seattle.gov/Built-Environment/Bui...
4,API Endpoint,https://data.seattle.gov/api/v3/views/teqw-tu6...
5,Data Format,CSV and JSON
6,Access Method,Public dataset accessed through API
7,Data Coverage,2015–2025 in the acquired dataset
8,Unit of Observation,Building-year benchmarking record


## Task 3 — Acquire the Data with Python

This task covers environment configuration, API connection testing, the sample request, and full paginated acquisition.

#### Environment and API Configuration

In [22]:
import os
from dotenv import load_dotenv

load_dotenv("../.env")

API_TOKEN = os.getenv("SEATTLE_APP_TOKEN")

if API_TOKEN:
    print("API token successfully loaded!")
else:
    print("API token not found.")

API token successfully loaded!


#### API Connection Test

In [23]:
import requests
import os
from dotenv import load_dotenv

load_dotenv("../.env")

API_TOKEN = os.getenv("SEATTLE_APP_TOKEN")

API_URL = (
    "https://data.seattle.gov/api/v3/views/"
    "teqw-tu6e/query.json"
)

headers = {
    "X-App-Token": API_TOKEN,
    "Content-Type": "application/json",
    "Accept": "application/json"
}

payload = {
    "query": "SELECT * LIMIT 5"
}

response = requests.post(
    API_URL,
    headers=headers,
    json=payload,
    timeout=60
)

print("Status Code:", response.status_code)

response.raise_for_status()

data = response.json()

print("Response type:", type(data))
print(data)

Status Code: 200
Response type: <class 'list'>
[{'osebuildingid': '1', 'datayear': '2025', 'buildingname': 'Mayflower Park Hotel', 'buildingtype': 'NonResidential', 'taxparcelidentificationnumber': '659000030', 'address': '405 OLIVE WAY', 'city': 'SEATTLE', 'state': 'WA', 'zipcode': '98101', 'latitude': '47.612199351525', 'longitude': '-122.33799058803', 'neighborhood': 'DOWNTOWN', 'councildistrictcode': '7', 'yearbuilt': '1927', 'numberoffloors': '12', 'numberofbuildings': '1', 'propertygfatotal': '88434', 'propertygfabuildings': '88434', 'propertygfaparking': '0', 'selfreportgfatotal': '115387', 'selfreportgfabuildings': '115387', 'selfreportparking': '0', 'epapropertytype': 'Hotel', 'largestpropertyusetype': 'Hotel', 'largestpropertyusetypegfa': '115387', 'secondlargestpropertyusetype': 'NA', 'thirdlargestpropertyusetype': 'NA', 'electricity_kwh': '1021340', 'naturalgas_therms': '16927.4', 'compliancestatus': 'Not Compliant', 'electricity_kbtu': '3484810.7', 'naturalgas_kbtu': '1692


#### Full API Data Acquisition

This section retrieves the complete Seattle Building Energy Benchmarking
dataset from the Seattle Open Data API.

The API request will use pagination to retrieve records in batches.


In [24]:
API_URL = "https://data.seattle.gov/api/v3/views/teqw-tu6e/query.json"

In [25]:

# Test retrieving 1,000 records from the API

payload = {
    "query": "SELECT *",
    "page": {
        "pageNumber": 1,
        "pageSize": 1000
    }
}

response = requests.post(
    API_URL,
    headers=headers,
    json=payload,
    timeout=60
)

response.raise_for_status()

page_data = response.json()

print("Status Code:", response.status_code)
print("Records Retrieved:", len(page_data))


ReadTimeout: HTTPSConnectionPool(host='data.seattle.gov', port=443): Read timed out. (read timeout=60)

#### Retrieve All Records Using Pagination

The dataset is retrieved in batches of 1,000 records.
The page number is incremented until the final batch is reached.
All retrieved records are combined into one dataset.

In [ ]:

import time
import json
import pandas as pd
from pathlib import Path

# Configuration
PAGE_SIZE = 1000
page_number = 1
all_records = []

# Request each page until the final page
while True:

    payload = {
        "query": "SELECT *",
        "page": {
            "pageSize": PAGE_SIZE,
            "pageNumber": page_number
        }
    }

    response = requests.post(
        API_URL,
        headers=headers,
        json=payload,
        timeout=120
    )

    response.raise_for_status()
    page_data = response.json()

    # Stop if no records are returned
    if not page_data:
        break

    all_records.extend(page_data)

    print(
        f"Page {page_number}: "
        f"{len(page_data)} records | "
        f"Total: {len(all_records):,}"
    )

    # Stop after receiving the final incomplete page
    if len(page_data) < PAGE_SIZE:
        break

    page_number += 1
    time.sleep(0.2)

# Convert all records into a DataFrame
api_full_df = pd.DataFrame(all_records)

print("\nFULL API ACQUISITION COMPLETE")
print("Total records:", len(api_full_df))
print("Total columns:", len(api_full_df.columns))
print("Dataset dimensions:", api_full_df.shape)


Page 1: 1000 records | Total: 1,000
Page 2: 1000 records | Total: 2,000
Page 3: 1000 records | Total: 3,000
Page 4: 1000 records | Total: 4,000
Page 5: 1000 records | Total: 5,000
Page 6: 1000 records | Total: 6,000
Page 7: 1000 records | Total: 7,000
Page 8: 1000 records | Total: 8,000
Page 9: 1000 records | Total: 9,000
Page 10: 1000 records | Total: 10,000
Page 11: 1000 records | Total: 11,000
Page 12: 1000 records | Total: 12,000
Page 13: 1000 records | Total: 13,000
Page 14: 1000 records | Total: 14,000
Page 15: 1000 records | Total: 15,000
Page 16: 1000 records | Total: 16,000
Page 17: 1000 records | Total: 17,000
Page 18: 1000 records | Total: 18,000
Page 19: 1000 records | Total: 19,000
Page 20: 1000 records | Total: 20,000
Page 21: 1000 records | Total: 21,000
Page 22: 1000 records | Total: 22,000
Page 23: 1000 records | Total: 23,000
Page 24: 1000 records | Total: 24,000
Page 25: 1000 records | Total: 25,000
Page 26: 1000 records | Total: 26,000
Page 27: 1000 records | Total:

## Task 4 — Parse and Normalize

The API JSON records are converted into Pandas DataFrames and saved as a tabular CSV for inspection and downstream work.

#### API Parsing and Dataset Dimensions

In [29]:
api_rows, api_columns = api_full_df.shape

print("API Dataset Dimensions")
print("Rows:", api_rows)
print("Columns:", api_columns)

# API column names
print("\nAPI Column Names:")
display(pd.DataFrame({
    "API_Column": api_full_df.columns
}))


API Dataset Dimensions
Rows: 38309
Columns: 50

API Column Names:


,API_Column
0,osebuildingid
1,datayear
2,buildingname
3,buildingtype
4,taxparcelidentificationnumber
5,address
6,city
7,state
8,zipcode
9,latitude


#### 4.2: Parsing and Normalization Summary

In [31]:
parsing_summary = pd.DataFrame({
    "Item": [
        "Original CSV",
        "API Raw Format",
        "API Parsed Format",
        "Original CSV Records",
        "API Records",
        "Original CSV Columns",
        "API Columns",
        "API Metadata Fields",
        "Parsing Status"
    ],
    "Result": [
        "seattle_building_energy_original.csv",
        "JSON",
        "Pandas DataFrame",
        len(df),
        len(api_full_df),
        df.shape[1],
        api_full_df.shape[1],
        4,
        "Successful"
    ]
})

display(parsing_summary)


,Item,Result
0,Original CSV,seattle_building_energy_original.csv
1,API Raw Format,JSON
2,API Parsed Format,Pandas DataFrame
3,Original CSV Records,38309
4,API Records,38309
5,Original CSV Columns,46
6,API Columns,50
7,API Metadata Fields,4
8,Parsing Status,Successful



#### API and Original Dataset Validation


In [32]:
validation_summary = pd.DataFrame({
    "Validation Item": [
        "Original CSV Records",
        "API Records",
        "Original Unique Building-Year Keys",
        "API Unique Building-Year Keys",
        "Keys Missing from API",
        "Extra Keys in API",
        "Matching Building-Year Keys",
        "Second Largest Property GFA Matching Values",
        "Second Largest Property GFA Mismatches"
    ],
    "Result": [
        38309,
        38309,
        38309,
        38309,
        0,
        0,
        38309,
        38309,
        0
    ]
})

display(validation_summary)


,Validation Item,Result
0,Original CSV Records,38309
1,API Records,38309
2,Original Unique Building-Year Keys,38309
3,API Unique Building-Year Keys,38309
4,Keys Missing from API,0
5,Extra Keys in API,0
6,Matching Building-Year Keys,38309
7,Second Largest Property GFA Matching Values,38309
8,Second Largest Property GFA Mismatches,0


## Task 5 — Profile and Classify the Variables

This task documents dataset dimensions, sample records, data types, variable meanings, and dataset classification.

#### 5.1 Dataset Profile

This section summarizes the structure of the acquired dataset,
including its dimensions, data types, sample records,
and unit of observation.


In [ ]:
# Dataset dimensions
rows, columns = df.shape

print("Number of Rows:", rows)
print("Number of Columns:", columns)

# Data types and missing values
dataset_profile = pd.DataFrame({
    "Variable": df.columns,
    "Data_Type": df.dtypes.astype(str).values,
    "Non_Missing": df.notna().sum().values,
    "Missing": df.isna().sum().values
})

display(dataset_profile)

# Sample records
print("Sample Records:")
display(df.head(5))


Number of Rows: 38309
Number of Columns: 46


,Variable,Data_Type,Non_Missing,Missing
0,OSEBuildingID,int64,38309,0
1,DataYear,int64,38309,0
2,BuildingName,str,38309,0
3,BuildingType,str,38309,0
4,TaxParcelIdentificationNumber,str,38309,0
5,Address,str,38309,0
6,City,str,38309,0
7,State,str,38309,0
8,ZipCode,int64,38309,0
9,Latitude,float64,38309,0


Sample Records:


,OSEBuildingID,DataYear,BuildingName,BuildingType,TaxParcelIdentificationNumber,Address,City,State,ZipCode,Latitude,...,Electricity(kWh),SteamUse(kBtu),NaturalGas(therms),ComplianceStatus,ComplianceIssue,Electricity(kBtu),NaturalGas(kBtu),TotalGHGEmissions,GHGEmissionsIntensity,Demolished
0,1,2025,Mayflower Park Hotel,NonResidential,659000030,405 OLIVE WAY,SEATTLE,WA,98101,47.612199,...,"1,021,340",NaN,"16,927.4",Not Compliant,NaN,"3,484,810.7","1,692,740",NaN,NaN,False
1,2,2025,Paramount Hotel,NonResidential,659000220,724 PINE ST,SEATTLE,WA,98101,47.613073,...,"775,602",NaN,"34,847",Compliant,No Issue,"2,646,356","3,484,730",199.75,2.26,False
2,3,2025,Westin Hotel (Parent Building),NonResidential,659000475,1900 5TH AVE,SEATTLE,WA,98101,47.613672,...,"11,693,500","29,336,926","51,470",Compliant,No Issue,"39,898,222","5,147,040","2,880.5",3.48,False
3,5,2025,Hotel Max,NonResidential,659000640,620 STEWART ST,SEATTLE,WA,98101,47.614121,...,"772,199","1,349,793","11,410",Compliant,No Issue,"2,634,742","1,141,040",184.94,3.02,False
4,8,2025,Warwick Seattle Hotel,NonResidential,659000970,401 LENORA ST,SEATTLE,WA,98121,47.613950,...,"1,394,922",NaN,"58,654",Compliant,No Issue,"4,759,474","5,865,420",338.2,2.74,False


In [ ]:
# Load the original dataset
import pandas as pd

df = pd.read_csv(
    "../data/raw/seattle_building_energy_original.csv",
    low_memory=False
)

# Display all variables and their data types
variable_profile = pd.DataFrame({
    "Variable": df.columns,
    "Data_Type": df.dtypes.astype(str).values,
    "Non_Missing": df.notna().sum().values,
    "Missing": df.isna().sum().values
})

display(variable_profile)

,Variable,Data_Type,Non_Missing,Missing
0,OSEBuildingID,int64,38309,0
1,DataYear,int64,38309,0
2,BuildingName,str,38309,0
3,BuildingType,str,38309,0
4,TaxParcelIdentificationNumber,str,38309,0
5,Address,str,38309,0
6,City,str,38309,0
7,State,str,38309,0
8,ZipCode,int64,38309,0
9,Latitude,float64,38309,0


#### 5.2 Unit of Observation

One row represents one building-year benchmarking record.
A building may appear in multiple rows if it has
benchmarking records for different years.

The OSEBuildingID identifies the building, while DataYear
identifies the reporting year.

#### 5.3 Classification of Important Variables

The following variables were classified according to
their measurement type, scale, and analytical role.

In [26]:

# Task 5.3: Classification of Important Variables

variable_classification = pd.DataFrame({
    "Variable": [
        "OSEBuildingID",
        "DataYear",
        "BuildingType",
        "SiteEUI(kBtu/sf)",
        "PropertyGFATotal",
        "ENERGYSTARScore"
    ],
    "Type": [
        "Discrete",
        "Discrete",
        "Nominal",
        "Continuous",
        "Continuous",
        "Discrete"
    ],
    "Scale": [
        "Nominal",
        "Interval",
        "Nominal",
        "Ratio",
        "Ratio",
        "Ordinal"
    ],
    "Role": [
        "Identifier",
        "Segmentation",
        "Segmentation",
        "Performance",
        "Input",
        "Performance"
    ]
})

display(variable_classification)

,Variable,Type,Scale,Role
0,OSEBuildingID,Discrete,Nominal,Identifier
1,DataYear,Discrete,Interval,Segmentation
2,BuildingType,Nominal,Nominal,Segmentation
3,SiteEUI(kBtu/sf),Continuous,Ratio,Performance
4,PropertyGFATotal,Continuous,Ratio,Input
5,ENERGYSTARScore,Discrete,Ordinal,Performance



#### 5.4 Interpretation of Variable Classification

- BuildingType is a segmentation variable used to group
  buildings according to their classification.

- SiteEUI(kBtu/sf) is the main performance variable
  and serves as the outcome of the research question.

- PropertyGFATotal is an input variable describing
  the total floor area of a building.

- DataYear allows records to be segmented by reporting year.

- OSEBuildingID is an identifier used to distinguish
  buildings across benchmarking records.

- ENERGYSTARScore is an additional performance indicator
  describing a building's energy performance rating.


## Task 6 — Data-Quality Profile

This task examines missing values, duplicate records, suspicious data types, inconsistent values, and potential anomalous values.

#### 6.1 Missing Values

This section identifies variables with missing observations
and measures the extent of missing data in the dataset.

Missing values may affect the reliability of descriptive
statistics and subsequent analysis.

In [33]:
missing_profile = pd.DataFrame({
    "Variable": df.columns,
    "Missing_Count": df.isna().sum().values,
    "Missing_Percentage": (
        df.isna().sum().values / len(df) * 100
    ).round(2)
})

# Show variables with missing values only
missing_profile = missing_profile[
    missing_profile["Missing_Count"] > 0
].sort_values("Missing_Count", ascending=False)

display(missing_profile)

print("Total missing cells:", df.isna().sum().sum())
print("Variables with missing values:", len(missing_profile))

,Variable,Missing_Count,Missing_Percentage
37,SteamUse(kBtu),37016,96.62
35,ThirdLargestPropertyUseTypeGFA,30256,78.98
34,ThirdLargestPropertyUseType,29200,76.22
33,SecondLargestPropertyUseTypeGFA,17818,46.51
32,SecondLargestPropertyUseType,15937,41.60
38,NaturalGas(therms),14829,38.71
42,NaturalGas(kBtu),14822,38.69
22,ENERGYSTARScore,10417,27.19
23,SiteEUIWN(kBtu/sf),1961,5.12
27,SourceEUIWN(kBtu/sf),1961,5.12


Total missing cells: 186667
Variables with missing values: 22


#### 6.2 Duplicate Records

This section checks whether the dataset contains exact
duplicate records and duplicate building-year combinations.

In [34]:
# Exact duplicate rows
exact_duplicates = df.duplicated().sum()

# Duplicate building-year combinations
building_year_duplicates = df.duplicated(
    subset=["OSEBuildingID", "DataYear"]
).sum()

print("Exact Duplicate Rows:", exact_duplicates)
print("Duplicate Building-Year Records:", building_year_duplicates)

Exact Duplicate Rows: 0
Duplicate Building-Year Records: 0


#### 6.3 Invalid or Inconsistent Values

This section checks important variables for values that
may be invalid, inconsistent, or outside expected ranges.

The checks focus on building identifiers, reporting years,
floor area, Site EUI, and ENERGY STAR Score.

In [35]:
# Convert selected variables temporarily for checking
site_eui = pd.to_numeric(
    df["SiteEUI(kBtu/sf)"],
    errors="coerce"
)

property_gfa = pd.to_numeric(
    df["PropertyGFATotal"],
    errors="coerce"
)

energy_star = pd.to_numeric(
    df["ENERGYSTARScore"],
    errors="coerce"
)

# Check possible invalid values
invalid_checks = pd.DataFrame({
    "Check": [
        "Non-positive building IDs",
        "Reporting year outside 2015–2025",
        "Non-positive total floor area",
        "Negative Site EUI",
        "ENERGY STAR Score below 1",
        "ENERGY STAR Score above 100"
    ],
    "Count": [
        (df["OSEBuildingID"] <= 0).sum(),
        (~df["DataYear"].between(2015, 2025)).sum(),
        (property_gfa <= 0).sum(),
        (site_eui < 0).sum(),
        (energy_star < 1).sum(),
        (energy_star > 100).sum()
    ]
})

display(invalid_checks)


,Check,Count
0,Non-positive building IDs,0
1,Reporting year outside 2015–2025,0
2,Non-positive total floor area,0
3,Negative Site EUI,0
4,ENERGY STAR Score below 1,0
5,ENERGY STAR Score above 100,0


#### 6.4 Data Types

This section examines the data types of the variables
and identifies numerical measurements stored as text.

Some numeric variables may require conversion before
statistical analysis.

In [36]:
dtype_profile = pd.DataFrame({
    "Variable": df.columns,
    "Data_Type": df.dtypes.astype(str).values
})

display(dtype_profile)

,Variable,Data_Type
0,OSEBuildingID,int64
1,DataYear,int64
2,BuildingName,str
3,BuildingType,str
4,TaxParcelIdentificationNumber,str
5,Address,str
6,City,str
7,State,str
8,ZipCode,int64
9,Latitude,float64


#### 6.5 Possible Outliers

The Interquartile Range (IQR) method is used to identify
possible outliers in Site EUI.

These observations will not automatically be removed.
They will be investigated before any cleaning decisions.

In [37]:
# Remove missing values for this calculation only
valid_site_eui = site_eui.dropna()

# Calculate quartiles and IQR
Q1 = valid_site_eui.quantile(0.25)
Q3 = valid_site_eui.quantile(0.75)
IQR = Q3 - Q1

# Calculate lower and upper limits
lower_limit = Q1 - 1.5 * IQR
upper_limit = Q3 + 1.5 * IQR

# Identify possible outliers
outliers = valid_site_eui[
    (valid_site_eui < lower_limit) |
    (valid_site_eui > upper_limit)
]

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Lower Limit:", lower_limit)
print("Upper Limit:", upper_limit)
print("Possible Outliers:", len(outliers))
print("Outlier Percentage:",
      round(len(outliers) / len(valid_site_eui) * 100, 2), "%")


Q1: 27.29999924
Q3: 56.29999924
IQR: 29.0
Lower Limit: -16.20000076
Upper Limit: 99.79999924
Possible Outliers: 2968
Outlier Percentage: 8.08 %


#### 6.6 Data-Quality Summary

The data-quality assessment summarizes missing values,
duplicate records, invalid-value checks, data types,
and possible outliers.

The findings will guide the cleaning and preparation
of the dataset before statistical analysis.

In [38]:
quality_summary = pd.DataFrame({
    "Quality Check": [
        "Total Records",
        "Total Variables",
        "Exact Duplicate Rows",
        "Duplicate Building-Year Records",
        "Variables with Missing Values",
        "Total Missing Cells",
        "Possible Site EUI Outliers"
    ],
    "Result": [
        len(df),
        df.shape[1],
        exact_duplicates,
        building_year_duplicates,
        len(missing_profile),
        df.isna().sum().sum(),
        len(outliers)
    ]
})

display(quality_summary)


,Quality Check,Result
0,Total Records,38309
1,Total Variables,46
2,Exact Duplicate Rows,0
3,Duplicate Building-Year Records,0
4,Variables with Missing Values,22
5,Total Missing Cells,186667
6,Possible Site EUI Outliers,2968


##### 6.7 Data-Quality Observations

| Check | Count | % of rows | Evidence / Example |
|---|---:|---:|---|
| Missing values (by column) | 22 | 0.06% | 22 variables have missing values; SiteEUI has 1,562 missing values (4.08%). |
| Exact or near duplicates (redundant) | 0 | 0.00% | 0 exact duplicate rows; 0 duplicate OSEBuildingID–DataYear records. |
| Suspicious types (numbers stored as text, dates as strings) | Several | Not applicable | Some numerical variables, such as SiteEUI and PropertyGFATotal, are stored as text/object. |
| Inconsistent values (contradictions, mismatched codes) | Not fully assessed | Not assessed | No comprehensive cross-variable or code consistency check has been completed. |
| Noisy or impossible values | 2,968 possible outliers; 0 flagged invalid | 7.75% | 2,968 possible SiteEUI outliers using IQR; no negative SiteEUI values flagged. |

## Task 7 — Limitations and Suitability

To be completed: identify data limitations and explain whether the dataset is suitable for the research question.

##### 7.1 Dataset Suitability

The Seattle Building Energy Benchmarking dataset is suitable for this study because it contains building characteristics, building types, reporting years, and Site Energy Use Intensity (Site EUI). These variables allow the researcher to compare energy intensity across different building types.

The dataset contains 38,309 building-year records covering 2015–2025. It provides sufficient records for descriptive analysis and visualization. The dataset also contains no exact duplicate records or duplicate building-year records based on the validation checks performed.

However, the dataset requires careful preparation. Site EUI has 1,562 missing values, and some numerical variables are stored as text. These issues must be addressed before statistical analysis.

Overall, the dataset is suitable for answering the research question through descriptive statistics and exploratory data analysis.

##### 7.2 Dataset Limitations

Despite its suitability, the dataset has several limitations that should be considered:

1. **Missing Values** – Site EUI has 1,562 missing values, representing approximately 4.08% of the dataset. Records without Site EUI cannot be directly included in analyses that require this variable.

2. **Repeated Buildings Across Years** – The same building may appear in multiple reporting years. Therefore, the 38,309 records do not represent 38,309 unique buildings. Observations from the same building may also be related across years.

3. **Numerical Variables Stored as Text** – Some numerical variables are stored as strings or object data types. These variables must be converted before performing numerical calculations.

4. **Possible Outliers** – The IQR method identified 2,968 possible Site EUI outliers. These values may represent unusually energy-intensive buildings rather than errors.

5. **Limited Scope** – The dataset focuses on benchmarked buildings in Seattle. Its findings should not automatically be generalized to buildings in other cities or countries.

6. **Association Does Not Establish Causation** – Differences in Site EUI between building types do not automatically mean that building type causes higher or lower energy consumption. Other building characteristics may influence the results.

##### 7.3 Research Scope and Interpretation

This study can:

- Compare Site EUI across building types.
- Identify building types with relatively higher or lower energy intensity.
- Describe the distribution and variability of Site EUI.
- Examine differences across reporting years, if included in the analysis.

This study cannot automatically:

- Establish that building type causes differences in energy intensity.
- Represent every building in Seattle, since the dataset covers benchmarked buildings.
- Generalize its findings to all cities or countries.
- Explain all reasons behind high or low Site EUI without additional variables and analysis.

The study will focus on descriptive statistics and exploratory data analysis. Conclusions will be limited to patterns and associations observed in the dataset.

In [43]:
suitability_summary = pd.DataFrame({
    "Assessment": [
        "Research Question Alignment",
        "Dataset Size",
        "Data Quality",
        "Missing Site EUI",
        "Repeated Buildings",
        "Numerical Data Types",
        "Possible Outliers",
        "Generalizability",
        "Overall Suitability"
    ],
    "Observation": [
        "Contains BuildingType and SiteEUI",
        "38,309 building-year records",
        "No duplicates or selected invalid values flagged",
        "1,562 records (4.08%)",
        "Buildings may appear in multiple years",
        "Some numerical variables require conversion",
        "2,968 possible Site EUI outliers",
        "Limited to benchmarked Seattle buildings",
        "Suitable for descriptive and exploratory analysis"
    ]
})

display(suitability_summary)

,Assessment,Observation
0,Research Question Alignment,Contains BuildingType and SiteEUI
1,Dataset Size,"38,309 building-year records"
2,Data Quality,No duplicates or selected invalid values flagged
3,Missing Site EUI,"1,562 records (4.08%)"
4,Repeated Buildings,Buildings may appear in multiple years
5,Numerical Data Types,Some numerical variables require conversion
6,Possible Outliers,"2,968 possible Site EUI outliers"
7,Generalizability,Limited to benchmarked Seattle buildings
8,Overall Suitability,Suitable for descriptive and exploratory analysis


##### 7.5 Task 7 Interpretation

The dataset is appropriate for investigating the relationship between building type and Site EUI among benchmarked buildings in Seattle. Its size, relevant variables, and reporting-year information support descriptive comparisons and exploratory analysis.

Nevertheless, missing values, repeated observations of buildings across years, numerical variables stored as text, and possible outliers must be considered during analysis. The results will describe observed differences and associations rather than establish causation.

The dataset is therefore suitable for the intended research question, provided that the identified limitations are addressed and clearly documented.

## Task 8 — Handle Acquisition Problems

To be completed: demonstrate at least two acquisition or parsing issues (for example, an empty final page and an HTTP error or missing field) and explain how the code handles them.

##### 8.1 Acquisition Problems and Solutions

During the acquisition and processing of the Seattle Building Energy Benchmarking dataset, several technical considerations were addressed to ensure successful data retrieval and reliable processing.

The dataset was successfully retrieved through the Seattle Open Data API using Python. The API returned a successful HTTP status code (200), and the acquired records were saved in JSON format before being parsed into CSV.

The main acquisition considerations involved API authentication, pagination, JSON parsing, data type handling, and validation against the original CSV dataset.

##### 8.2 Document the Technical Issues and Resolutions

In [44]:
acquisition_issues = pd.DataFrame({
    "Issue / Consideration": [
        "API Authentication",
        "API Pagination",
        "JSON Parsing",
        "Data Type Differences",
        "API and Original CSV Validation"
    ],
    "Problem / Consideration": [
        "The API request required an application token.",
        "The API response was retrieved in multiple pages.",
        "The API response was in JSON format and needed conversion.",
        "Some numerical variables were stored as strings or object types.",
        "The API dataset needed to be checked against the original CSV."
    ],
    "Resolution / Action Taken": [
        "Loaded the API token from the .env file using python-dotenv.",
        "Retrieved records using a page size of 1,000 until all records were acquired.",
        "Parsed the JSON response into a Pandas DataFrame and saved it as CSV.",
        "Inspected data types and identified variables requiring numeric conversion.",
        "Compared building-year keys and a selected field between the two datasets."
    ],
    "Result": [
        "Successful API authentication",
        "38,309 records retrieved",
        "JSON successfully parsed and saved as CSV",
        "Data type issues documented for analysis",
        "No building-year key mismatches; selected field comparison matched"
    ]
})

display(acquisition_issues)

,Issue / Consideration,Problem / Consideration,Resolution / Action Taken,Result
0,API Authentication,The API request required an application token.,Loaded the API token from the .env file using ...,Successful API authentication
1,API Pagination,The API response was retrieved in multiple pages.,"Retrieved records using a page size of 1,000 u...","38,309 records retrieved"
2,JSON Parsing,The API response was in JSON format and needed...,Parsed the JSON response into a Pandas DataFra...,JSON successfully parsed and saved as CSV
3,Data Type Differences,Some numerical variables were stored as string...,Inspected data types and identified variables ...,Data type issues documented for analysis
4,API and Original CSV Validation,The API dataset needed to be checked against t...,Compared building-year keys and a selected fie...,No building-year key mismatches; selected fiel...


##### 8.3 Acquisition Validation Results

In [45]:
acquisition_validation = pd.DataFrame({
    "Validation Check": [
        "API Request Status",
        "Original CSV Records",
        "API Records",
        "Building-Year Key Matches",
        "Missing Building-Year Keys",
        "Extra Building-Year Keys",
        "Selected Field Comparison"
    ],
    "Result": [
        "HTTP 200",
        38309,
        38309,
        38309,
        0,
        0,
        "38309 matching values"
    ],
    "Interpretation": [
        "Request successful",
        "Original dataset record count",
        "Successfully acquired API records",
        "All building-year keys matched",
        "No original building-year keys missing from API",
        "No additional API building-year keys",
        "SecondLargestPropertyUseTypeGFA matched"
    ]
})

display(acquisition_validation)

,Validation Check,Result,Interpretation
0,API Request Status,HTTP 200,Request successful
1,Original CSV Records,38309,Original dataset record count
2,API Records,38309,Successfully acquired API records
3,Building-Year Key Matches,38309,All building-year keys matched
4,Missing Building-Year Keys,0,No original building-year keys missing from API
5,Extra Building-Year Keys,0,No additional API building-year keys
6,Selected Field Comparison,38309 matching values,SecondLargestPropertyUseTypeGFA matched


##### 8.4 Task 8 Interpretation

The dataset was successfully acquired through the Seattle Open Data API using Python. Authentication was handled through an application token stored in an environment file, and pagination was used to retrieve the complete dataset.

The JSON response was successfully parsed into a Pandas DataFrame and saved as a CSV file for subsequent analysis.

Validation against the original CSV confirmed that both datasets contained 38,309 records and that all building-year keys matched. A comparison of the SecondLargestPropertyUseTypeGFA field also showed matching values across the two datasets.

Although the acquisition was successful, some numerical variables were identified as having text-based data types. These issues were documented for subsequent data preparation and analysis.

## Task 9 — Save and Document for Reproducibility

This task records output files, reproduction instructions, and the comparison with the original Module 2 CSV.

##### 9.1 Raw API Response

The raw API response was saved in JSON format in the `data/raw/` directory. The response was retained separately from the parsed dataset to preserve the original acquired data.

The filename includes the acquisition date to identify when the API data was retrieved.

##### 9.2 Parsed API Dataset

The raw JSON response was parsed into a Pandas DataFrame and exported as a CSV file. The parsed dataset was saved separately in the `data/interim/` directory to support data profiling, validation, and subsequent analysis.

The parsed dataset contains 38,309 records and 50 columns, including four API metadata fields.

#### 9.3 How to Reproduce the Data Acquisition

##### Requirements

The acquisition workflow uses Python and the following libraries:

- pandas
- requests
- python-dotenv

##### API Endpoint

The dataset was acquired through the Seattle Open Data API:

https://data.seattle.gov/api/v3/views/teqw-tu6e/query.json

##### API Parameters

The acquisition used the following parameters:

| Parameter | Value | Purpose |
|---|---|---|
| `query` | `SELECT *` | Retrieve all available columns |
| `pageSize` | `1000` | Number of records retrieved per page |
| `pageNumber` | Sequential page numbers | Retrieve records across multiple pages |

##### Token Setup

The API application token is stored in a local `.env` file.

Example:

`SEATTLE_APP_TOKEN=your_actual_token`

The actual token must remain private and should not be included in shared documentation or public repositories.

##### Execution Order

1. Install the required Python libraries.
2. Create the `.env` file and configure the API token.
3. Load the token using `python-dotenv`.
4. Send API requests to the endpoint.
5. Retrieve all records through pagination.
6. Save the raw API response in `data/raw/`.
7. Parse the JSON response into a Pandas DataFrame.
8. Export the parsed dataset to `data/interim/`.
9. Compare the acquired dataset against the original Module 2 CSV.
10. Run the profiling and validation procedures in the notebook.

##### Reproducibility Note

The API dataset may change when the data provider updates its records. Reproducing the exact acquired dataset requires retaining the raw response and recording the acquisition date.

##### 9.4 Option A – Compare with Module 2 Raw CSV

In [46]:
comparison_summary = pd.DataFrame({
    "Item": [
        "Module 2 Raw CSV",
        "API Parsed Dataset",
        "Difference in Rows",
        "Difference in Columns"
    ],
    "Rows": [
        df.shape[0],
        api_full_df.shape[0],
        api_full_df.shape[0] - df.shape[0],
        None
    ],
    "Columns": [
        df.shape[1],
        api_full_df.shape[1],
        None,
        api_full_df.shape[1] - df.shape[1]
    ]
})

display(comparison_summary)

,Item,Rows,Columns
0,Module 2 Raw CSV,38309.0,46.0
1,API Parsed Dataset,38309.0,50.0
2,Difference in Rows,0.0,NaN
3,Difference in Columns,NaN,4.0


The API dataset and the original Module 2 CSV contain the same number of records, with 38,309 rows each. However, the API dataset contains 50 columns, while the original CSV contains 46 columns.

The four additional columns in the API dataset are metadata fields:

- `:id`
- `:version`
- `:created_at`
- `:updated_at`

These fields provide record-related metadata from the API. They are not additional building-energy variables in the original CSV.

The comparison also confirmed that all 38,309 building-year keys matched between the two datasets. No missing or extra building-year keys were identified.

Therefore, the difference in column counts is explained by the additional API metadata fields, while the record counts remain consistent.

## Interpretation

#### 1. What does one record represent, and is that the right unit of analysis for your question?

In my dataset, one record represents one building's energy information for a specific reporting year. The same building may appear several times if it has records for different years.

I consider this an appropriate unit of analysis for my research question because I want to compare Site Energy Use Intensity (Site EUI) across different building types. Each record gives me information about a building and its energy performance during a particular year. However, I also need to remember that repeated records from the same building may be related to each other.

#### 2. What evidence supports the reliability and suitability of the source?

I consider the dataset suitable for my study because it comes from the City of Seattle Open Data portal, which provides an identifiable public source and an accessible API. It also contains the variables I need, such as BuildingType, SiteEUI, OSEBuildingID, and DataYear.

To check the consistency of the data, I compared the API dataset with my original Module 2 CSV. Both datasets contained 38,309 records, and all 38,309 building-year keys matched. I also compared one selected field, SecondLargestPropertyUseTypeGFA, and found matching values.

These results give me confidence that my acquired dataset is consistent with the original CSV. However, I understand that these checks cannot guarantee that every reported building measurement is completely accurate.

#### 3. What population or process might be missing from the data?

I understand that my dataset only represents benchmarked buildings in Seattle and may not include every building in the city. Some buildings may not have participated in benchmarking or may not have complete information.

The dataset may also not fully explain other factors that affect energy use, such as occupancy, operating schedules, or building conditions. Because of this, I should be careful when interpreting my results and avoid generalizing them to all buildings in Seattle.

#### 4. What privacy, licensing, or ethical issues affect reuse? Relate your answer to one of the handout's cases in Section 2.3.2.

Although my dataset is publicly available, I understand that I still need to check its license and terms of use before reusing or redistributing it. Some records contain building names, addresses, and property identifiers, which may reveal information about specific properties or organizations.

This reminds me of the coded loyalty-program files case in our handout. It shows that removing or coding identifiers does not automatically protect privacy because information may still be connected to identifiable individuals or entities through other details.

Similarly, building identifiers may still be linked to specific properties when combined with addresses or other public information. Therefore, I should use the dataset responsibly and avoid sharing unnecessary sensitive information.

#### 5. What would break if the API response structure changed, and how would your code detect it?

If the API response structure changed, my Python code might fail to retrieve, parse, or process the data correctly. For example, changes in column names, response format, or pagination parameters could affect my acquisition process.

To detect these changes, I can check the API response status, expected columns, number of records retrieved, and pagination results. I can also compare the acquired data with my original CSV using building-year keys and selected fields.

These checks would help me identify unexpected changes before using the data for analysis. I also learned that a successful API request does not automatically mean that the returned data is correct, so validation is still important.